In [1]:
# Import numpy for handling tables and arrays
import numpy as np

# Import gymnasium for the Mountain Car environment
import gymnasium as gym

# Load the Mountain Car environment
env = gym.make("MountainCar-v0")

# Number of bins for discretizing the continuous state space
num_bins = 20

# Create evenly spaced bins for position and velocity
pos_bins = np.linspace(-1.2, 0.6, num_bins)
vel_bins = np.linspace(-0.07, 0.07, num_bins)

# ---------------------------------------------------
# Step 1: Initialize Two Q-Tables
# ---------------------------------------------------

# Q1 and Q2 store the Q-values for each state-action pair
Q1 = np.zeros((num_bins, num_bins, 3))
Q2 = np.zeros((num_bins, num_bins, 3))

# Hyperparameters
alpha = 0.1      # Learning rate
gamma = 0.99     # Discount factor
epsilon = 0.1    # Exploration rate

# ---------------------------------------------------
# Helper Function: Discretize Continuous State
# ---------------------------------------------------

def get_discretized_state(state):
    # Extract position and velocity
    pos, vel = state

    # Find corresponding bin indices
    pos_idx = np.digitize(pos, pos_bins) - 1
    vel_idx = np.digitize(vel, vel_bins) - 1

    # Keep indices within valid range
    pos_idx = np.clip(pos_idx, 0, num_bins - 1)
    vel_idx = np.clip(vel_idx, 0, num_bins - 1)

    return int(pos_idx), int(vel_idx)

# ---------------------------------------------------
# Step 2: Train the Agent
# ---------------------------------------------------

for episode in range(1000):

    # Reset environment
    raw_state, info = env.reset()
    state = get_discretized_state(raw_state)

    done = False

    while not done:

        # Epsilon-Greedy Action Selection
        if np.random.rand() < epsilon:
            # Explore
            action = env.action_space.sample()
        else:
            # Exploit using both Q-tables
            action = np.argmax(
                Q1[state[0], state[1]] +
                Q2[state[0], state[1]]
            )

        # Take action
        next_raw_state, reward, terminated, truncated, _ = env.step(action)

        next_state = get_discretized_state(next_raw_state)
        done = terminated or truncated

        # ---------------------------------------------------
        # Step 3: Double Q-Learning Update
        # ---------------------------------------------------

        if np.random.rand() < 0.5:

            # Update Q1 using Q2 for evaluation
            best_next_action = np.argmax(
                Q1[next_state[0], next_state[1]]
            )

            target = reward + gamma * Q2[
                next_state[0],
                next_state[1],
                best_next_action
            ]

            Q1[state[0], state[1], action] += alpha * (
                target - Q1[state[0], state[1], action]
            )

        else:

            # Update Q2 using Q1 for evaluation
            best_next_action = np.argmax(
                Q2[next_state[0], next_state[1]]
            )

            target = reward + gamma * Q1[
                next_state[0],
                next_state[1],
                best_next_action
            ]

            Q2[state[0], state[1], action] += alpha * (
                target - Q2[state[0], state[1], action]
            )

        # Move to the next state
        state = next_state

# ---------------------------------------------------
# Display Results
# ---------------------------------------------------

print("Double Q-Tables trained successfully!")

print("\nSample Q-values for starting position [-0.5, 0.0]:")

start_idx = get_discretized_state([-0.5, 0.0])

print("Table 1:", Q1[start_idx[0], start_idx[1]])
print("Table 2:", Q2[start_idx[0], start_idx[1]])

Double Q-Tables trained successfully!

Sample Q-values for starting position [-0.5, 0.0]:
Table 1: [-42.57413586 -42.87632828 -42.78071979]
Table 2: [-42.62271137 -42.91575079 -42.98080026]
